# MIMIC Deterioration Pipeline - Getting Started Guide

**Welcome!** This notebook will guide you through setting up and running the MIMIC deterioration pipeline for the first time.

---

## What This Notebook Covers

1. **Part 1: Prerequisites & Setup** - Verify requirements and environment
2. **Part 2: Database Setup** - Connect to PostgreSQL and load MIMIC-IV data
3. **Part 3: Pipeline Execution** - Run the feature extraction pipeline
4. **Part 4: Dataset Generation** - Create analysis-ready datasets
5. **Part 5: Quick Data Exploration** - Verify outputs and explore results

---

## Before You Begin

**Required:**
- MIMIC-IV database loaded in PostgreSQL
- Python 3.10+ with required packages
- PostgreSQL 15+ running locally or remotely
- MIMIC-IV physionet credentials

**Estimated Time:** 30-60 minutes (depending on data size)

---

# PART 1: Prerequisites & Setup Verification

Let's verify that everything is installed and configured correctly.

In [2]:
# Check Python version (should be 3.10+)
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

if sys.version_info < (3, 10):
    print("⚠️  WARNING: Python 3.10+ is recommended")
else:
    print("✓ Python version OK")

Python version: 3.14.1 (tags/v3.14.1:57e0d17, Dec  2 2025, 14:05:07) [MSC v.1944 64 bit (AMD64)]
Python executable: c:\Users\Lab\Desktop\MIMICc_Deterioration_Pipeline - Copy\.venv\Scripts\python.exe
✓ Python version OK


In [9]:
# Check required packages
import importlib

required_packages = [
    'pandas',
    'numpy',
    'psycopg2',
    'yaml',
    'jinja2',
    'sklearn'
]

print("Checking required packages...\n")
missing = []

for pkg in required_packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'unknown')
        print(f"✓ {pkg:15s} {version}")
    except ImportError:
        print(f"✗ {pkg:15s} NOT FOUND")
        missing.append(pkg)

if missing:
    print(f"\n⚠️  Missing packages: {', '.join(missing)}")
    print("Install with: pip install -r requirements.txt")
else:
    print("\n✓ All required packages installed")

Checking required packages...

✓ pandas          3.0.0
✓ numpy           2.4.2
✓ psycopg2        2.9.11 (dt dec pq3 ext lo64)
✓ yaml            6.0.3
✓ jinja2          3.1.6
✓ sklearn         1.8.0

✓ All required packages installed


In [8]:
#pip install jinja2

In [7]:
#pip install scikit-learn

In [ ]:
#cd c:\Users\Lab\Desktop\MIMICc_Deterioration_Pipeline - Copy

c:\Users\Lab\Desktop\MIMICc_Deterioration_Pipeline - Copy


In [15]:
# Verify working directory and project structure
import os
from pathlib import Path

print(f"Current directory: {os.getcwd()}\n")

# Check for essential directories and files
required_items = [
    ('Directory', 'src'),
    ('Directory', 'sql'),
    ('Directory', 'config'),
    ('File', 'config/config.yaml'),
    ('File', 'requirements.txt'),
]

print("Checking project structure...\n")
all_present = True

for item_type, item_path in required_items:
    exists = Path(item_path).exists()
    status = "✓" if exists else "✗"
    print(f"{status} {item_type:10s} {item_path}")
    if not exists:
        all_present = False

if all_present:
    print("\n✓ Project structure looks good")
else:
    print("\n⚠️  Some files/directories are missing. Make sure you're in the project root.")

Current directory: c:\Users\Lab\Desktop\MIMICc_Deterioration_Pipeline - Copy

Checking project structure...

✓ Directory  src
✓ Directory  sql
✓ Directory  config
✓ File       config/config.yaml
✓ File       requirements.txt

✓ Project structure looks good


---

# PART 2: Database Setup & Configuration

Now we'll configure the database connection and verify MIMIC-IV data is accessible.

## Step 2.1: Set Database Password

**IMPORTANT:** Configure your PostgreSQL password.

Edit the `PGPASSWORD` variable below with your actual PostgreSQL password, then run the cell:

In [ ]:
import os

# EDIT THIS: Set your PostgreSQL password
PGPASSWORD = "---"  # Replace with your actual password

# Set environment variable
os.environ['PGPASSWORD'] = PGPASSWORD
print("✓ PostgreSQL password configured for this session")

✓ PostgreSQL password configured for this session


## Step 2.2: Load Configuration

Load the pipeline configuration from `config/config.yaml`:

In [17]:
from src.utils import load_yaml
import yaml

# Load configuration
cfg = load_yaml("config/config.yaml")

print("Configuration loaded:\n")
print(yaml.dump(cfg, default_flow_style=False, sort_keys=False))

Configuration loaded:

db:
  host: localhost
  port: 5432
  name: mimiciv
  user: postgres
  password_env: PGPASSWORD
schemas:
  ed: mimiciv_ed
  hosp: mimiciv_hosp
  icu: mimiciv_icu
  ecg: mimiciv_ecg
tables:
  base_ed_cohort: tmp_base_ed_cohort
  event_log: tmp_ed_event_log
  outcomes: tmp_ed_outcomes
  features_w1: tmp_features_w1
  features_w6: tmp_features_w6
  features_w24: tmp_features_w24
  ecg_record_list: mimiciv_ecg.record_list
  ecg_machine_measurements: mimiciv_ecg.machine_measurements
  ecg_features_w1: tmp_ecg_features_w1
  ecg_features_w6: tmp_ecg_features_w6
cohort:
  min_age: 18
  exclude_age_over: 89
  require_hadm_id: true
pipeline:
  verbose: true
  log_queries: false
  validate_after_each_step: true
ecg:
  enabled: true



### 📝 Configuration Check

**Verify these settings match your setup:**
- `db.host` - Your PostgreSQL server (usually `localhost`)
- `db.port` - PostgreSQL port (usually `5432`)
- `db.name` - Your MIMIC-IV database name
- `db.user` - Your PostgreSQL username
- `schemas.ed`, `schemas.hosp`, `schemas.icu` - MIMIC-IV schema names

**If anything is incorrect, edit `config/config.yaml` and re-run the cell above.**

## Step 2.3: Test Database Connection

In [18]:
from src.db import get_conn
import psycopg2

print("Testing database connection...\n")

try:
    # Attempt connection
    conn = get_conn(cfg)
    
    # Test query
    cur = conn.cursor()
    cur.execute("SELECT version();")
    version = cur.fetchone()[0]
    
    print("✓ Connection successful!\n")
    print(f"PostgreSQL version:\n{version[:100]}...\n")
    
    # Check schemas exist
    cur.execute("""
        SELECT schema_name 
        FROM information_schema.schemata 
        WHERE schema_name IN ('mimiciv_ed', 'mimiciv_hosp', 'mimiciv_icu','mimiciv_ecg')
        ORDER BY schema_name
    """)
    
    schemas = [row[0] for row in cur.fetchall()]
    print("MIMIC-IV schemas found:")
    for schema in schemas:
        print(f"  ✓ {schema}")
    
    if len(schemas) < 4:    #change to 3 if not using ECG data ..
        print("\n⚠️  WARNING: Some MIMIC-IV schemas are missing!")
        print("Expected: mimiciv_ed, mimiciv_hosp, mimiciv_icu, mimiciv_ecg")
    
    cur.close()
    
except psycopg2.OperationalError as e:
    print(f"✗ Connection failed: {e}\n")
    print("Troubleshooting:")
    print("  1. Check PostgreSQL is running")
    print("  2. Verify config.yaml has correct host/port/database")
    print("  3. Verify password was entered correctly")
    print("  4. Check firewall settings if using remote database")
    conn = None

except Exception as e:
    print(f"✗ Unexpected error: {e}")
    conn = None

Testing database connection...

✓ Connection successful!

PostgreSQL version:
PostgreSQL 18.1 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit...

MIMIC-IV schemas found:
  ✓ mimiciv_ecg
  ✓ mimiciv_ed
  ✓ mimiciv_hosp
  ✓ mimiciv_icu


## Step 2.4: Verify MIMIC-IV Data

In [19]:
if conn is None:
    print("⚠️  Cannot verify data - connection not established")
else:
    print("Checking MIMIC-IV tables...\n")
    
    # Tables we need from MIMIC-IV
    required_tables = [
        (cfg['schemas']['ed'], 'edstays'),
        (cfg['schemas']['ed'], 'vitalsign'),
        (cfg['schemas']['ed'], 'triage'),
        (cfg['schemas']['hosp'], 'patients'),
        (cfg['schemas']['hosp'], 'admissions'),
        (cfg['schemas']['hosp'], 'labevents'),
        (cfg['schemas']['icu'], 'icustays'),
    ]
    
    cur = conn.cursor()
    all_present = True
    
    for schema, table in required_tables:
        cur.execute(f"""
            SELECT COUNT(*) as row_count
            FROM {schema}.{table}
        """)
        
        try:
            count = cur.fetchone()[0]
            print(f"✓ {schema:15s}.{table:20s} {count:>10,} rows")
        except Exception as e:
            print(f"✗ {schema:15s}.{table:20s} NOT FOUND")
            all_present = False
    
    cur.close()
    
    if all_present:
        print("\n✓ All required MIMIC-IV tables present")
    else:
        print("\n⚠️  Some tables are missing. Verify MIMIC-IV is fully loaded.")

Checking MIMIC-IV tables...

✓ mimiciv_ed     .edstays                 425,087 rows
✓ mimiciv_ed     .vitalsign             1,564,610 rows
✓ mimiciv_ed     .triage                  425,087 rows
✓ mimiciv_hosp   .patients                364,627 rows
✓ mimiciv_hosp   .admissions              546,028 rows
✓ mimiciv_hosp   .labevents            158,374,764 rows
✓ mimiciv_icu    .icustays                 94,458 rows

✓ All required MIMIC-IV tables present


## Step 2.5: Check ECG Data (Optional)

ECG features are available for **W1 (1-hour)** and **W6 (6-hour)** windows only.

Let's check if ECG data is loaded:

In [20]:
if conn is None:
    print("⚠️  Cannot check ECG data - connection not established")
else:
    print("Checking ECG data availability...\n")
    
    cur = conn.cursor()
    
    # Check if ECG schema and tables exist
    try:
        cur.execute("""
            SELECT schema_name 
            FROM information_schema.schemata 
            WHERE schema_name = 'mimiciv_ecg'
        """)
        
        ecg_schema_exists = cur.fetchone() is not None
        
        if ecg_schema_exists:
            print("✓ ECG schema found: mimiciv_ecg\n")
            
            # Check ECG tables
            ecg_tables = ['record_list', 'machine_measurements']
            ecg_available = True
            
            for table in ecg_tables:
                try:
                    cur.execute(f"SELECT COUNT(*) FROM mimiciv_ecg.{table}")
                    count = cur.fetchone()[0]
                    print(f"  ✓ mimiciv_ecg.{table:25s} {count:>10,} rows")
                except Exception:
                    print(f"  ✗ mimiciv_ecg.{table:25s} NOT FOUND")
                    ecg_available = False
            
            if ecg_available:
                print("\n✓ ECG data is available")
                print("  → ECG features will be generated for W1 and W6 windows")
                print("  → Scripts: 33_ecg_features_w1.sql, 34_ecg_features_w6.sql")
            else:
                print("\n⚠️  ECG tables incomplete")
        else:
            print("⚠️  ECG schema not found (mimiciv_ecg)")
            print("\nECG features will NOT be available.")
            print("This is optional - pipeline will run without ECG features.")
            print("\nTo add ECG data:")
            print("  1. See PART0_DATABASE_SETUP.ipynb")
            print("  2. Download MIMIC-IV-ECG from Physionet")
            print("  3. Load ECG tables into mimiciv_ecg schema")
    
    except Exception as e:
        print(f"Error checking ECG data: {e}")
    
    finally:
        cur.close()

Checking ECG data availability...

✓ ECG schema found: mimiciv_ecg

  ✓ mimiciv_ecg.record_list                  800,035 rows
  ✓ mimiciv_ecg.machine_measurements         800,035 rows

✓ ECG data is available
  → ECG features will be generated for W1 and W6 windows
  → Scripts: 33_ecg_features_w1.sql, 34_ecg_features_w6.sql


---

## Summary

Let's review what we've verified:

In [21]:
# Print summary of setup status
print("="*70)
print("SETUP VERIFICATION SUMMARY")
print("="*70 + "\n")

# Check Python
import sys
python_ok = sys.version_info >= (3, 10)
print(f"{'✓' if python_ok else '✗'} Python {sys.version_info.major}.{sys.version_info.minor}")

# Check packages
try:
    import pandas, numpy, psycopg2, yaml, jinja2, sklearn
    packages_ok = True
except ImportError:
    packages_ok = False
print(f"{'✓' if packages_ok else '✗'} Required packages")

# Check database
db_ok = conn is not None
print(f"{'✓' if db_ok else '✗'} Database connection")

# Check schemas
if db_ok:
    print(f"{'✓' if len(schemas) >= 4 else '✗'} MIMIC-IV schemas ({len(schemas)}/4)")
    print(f"{'✓' if all_present else '✗'} Required tables")
    
    # ECG status
    try:
        cur = conn.cursor()
        cur.execute("SELECT COUNT(*) FROM mimiciv_ecg.record_list")
        ecg_count = cur.fetchone()[0]
        cur.close()
        print(f"✓ ECG data ({ecg_count:,} records)")
    except:
        print(f"⚠️  ECG data (not available - optional)")

print("\n" + "="*70)

if python_ok and packages_ok and db_ok and all_present:
    print("🎉 ALL CHECKS PASSED - Ready to run pipeline!")
    print("\nNext step: Continue to Part 3 (Pipeline Execution)")
else:
    print("⚠️  Some checks failed - please review and fix issues above")
    
print("="*70)

SETUP VERIFICATION SUMMARY

✓ Python 3.14
✓ Required packages
✓ Database connection
✓ MIMIC-IV schemas (4/4)
✓ Required tables
✓ ECG data (800,035 records)

🎉 ALL CHECKS PASSED - Ready to run pipeline!

Next step: Continue to Part 3 (Pipeline Execution)


---

**✋ CHECKPOINT:** Before proceeding to Part 3, ensure:
- ✓ Python 3.10+ installed
- ✓ Required packages installed
- ✓ Database connection successful
- ✓ All MIMIC-IV schemas present (mimiciv_ed, mimiciv_hosp, mimiciv_icu)
- ✓ All required tables found with data
- ⚠️  ECG data optional (only for W1/W6 ECG features)

If everything above passed, continue to **Part 3: Pipeline Execution**!

---